# 4. Ragged stratification

Real models are not rectangular. Severity applies to the infectious, not to the
susceptible. Treatment line applies to the diagnosed. Strain applies to
everything except the recovered, depending on the question.

summer4 makes this **ragged** structure first class: a property may be absent on
some compartments, and the query algebra knows the difference between *false*
and *not applicable*.

## Stratifying a subset

Pass a selector as `where=`. Only compartments where it is **true** are split.

In [ ]:
from summer4 import Property, PropertyMap

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
severity = Property("severity", ("mild", "severe"))

pmap = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(severity, where=state["I"])
)

assert pmap.size == 12  # 3 S + 3x2 I + 3 R
pmap

The infectious compartments gained a severity axis; the rest kept their single
row. `to_dicts()` shows the hole directly — absent properties are simply not
present in the dictionary.

In [ ]:
rows = pmap.to_dicts()

assert rows[0] == {"state": "S", "age": "0-4"}
assert rows[3] == {"state": "I", "age": "0-4", "severity": "mild"}
assert "severity" not in rows[0]

## Seeing the holes

The underlying table is an `int16` matrix of trait codes where `-1` marks
"property does not apply". Plotting it makes the raggedness obvious.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

codes = np.asarray(pmap.codes)
display = np.where(codes < 0, np.nan, codes.astype(float))

fig, ax = plt.subplots(figsize=(4.5, 5))
ax.imshow(display, cmap="viridis", aspect="auto")
ax.set_xticks(range(pmap.n_properties))
ax.set_xticklabels([p.name for p in pmap.properties], rotation=30, ha="right")
ax.set_yticks(range(pmap.size))
ax.set_yticklabels(pmap.labels(), fontsize=7)
ax.set_title("Trait codes (blank = property absent)")
for row in range(pmap.size):
    for col in range(pmap.n_properties):
        if codes[row, col] < 0:
            ax.text(col, row, "-", ha="center", va="center", color="0.4")
fig.tight_layout()
plt.show()

## Three-valued logic

Every selector evaluates internally to an `int8` array with three values:

| Value | Meaning |
|---|---|
| `1` | true |
| `0` | unknown — the property does not apply here |
| `-1` | false |

`select` and `mask` keep only **true**. This is [Kleene's strong three-valued
logic](https://en.wikipedia.org/wiki/Three-valued_logic): negation flips true
and false but leaves unknown alone.

The consequence is the single most important rule in this chapter:

```{admonition} Neither a trait nor its negation reaches an absent compartment
:class: important

`severity["mild"]` and `~severity["mild"]` both **exclude** the compartments
that have no severity axis at all.
```

In [ ]:
mild = set(pmap.select(severity["mild"]).tolist())
not_mild = set(pmap.select(~severity["mild"]).tolist())
absent = set(pmap.select(severity.absent()).tolist())

assert mild == {3, 5, 7}
assert not_mild == {4, 6, 8}
assert absent == {0, 1, 2, 9, 10, 11}

# A clean three-way partition of every compartment.
assert mild | not_mild | absent == set(range(pmap.size))
assert mild.isdisjoint(not_mild) and mild.isdisjoint(absent)

This is what you want. If `~severity["mild"]` silently included every
susceptible compartment, a "non-mild cases" flow would quietly pick up the whole
uninfected population the first time somebody made severity ragged.

## Reaching the unstratified compartments

`present()` and `absent()` are **two-valued**: they answer a question about the
table, not about the trait, so they never return unknown.

In [ ]:
assert set(pmap.select(severity.present()).tolist()) == set(pmap.select(state["I"]).tolist())
assert set(pmap.select(severity.absent()).tolist()) == set(
    pmap.select(state["S"] | state["R"]).tolist()
)

assert pmap.select(severity.present()).size + pmap.select(severity.absent()).size == pmap.size

### The idiom for "mild, or not applicable"

Combining a trait with `absent()` is how you write "the compartments where
severity is mild, *and* the ones where the question does not arise" — for
example when aggregating a total that must cover the whole population.

In [ ]:
mild_or_na = pmap.select(severity["mild"] | severity.absent())
assert mild_or_na.size == 3 + 6

everyone = pmap.select(severity.present() | severity.absent())
assert everyone.size == pmap.size

## The truth tables, computed

Rather than taking the rules on trust, here they are evaluated by the library.
We build a nine-row map in which two ragged properties `a` and `b` between them
realise every combination of true, false and unknown.

In [ ]:
has_a = Property("has_a", ("with", "without"))
has_b = Property("has_b", ("with", "without"))
a = Property("a", ("yes", "no"))
b = Property("b", ("yes", "no"))

grid = (
    PropertyMap.from_property(has_a)
    .stratify(has_b)
    .stratify(a, where=has_a["with"])
    .stratify(b, where=has_b["with"])
)
assert grid.size == 9  # 3 states of a  x  3 states of b


def state_of(pm, prop, sel):
    """Return 'T'/'F'/'U' per compartment for a single-property selector."""
    true = pm.mask(sel)
    unknown = pm.mask(prop.absent())
    return np.where(true, "T", np.where(unknown, "U", "F"))


a_state = state_of(grid, a, a["yes"])
b_state = state_of(grid, b, b["yes"])
assert sorted(zip(a_state, b_state)) == sorted(
    (x, y) for x in "TFU" for y in "TFU"
)

In [ ]:
def table(pm, sel, a_state, b_state):
    """Render a 3x3 truth table for a binary selector expression."""
    true = pm.mask(sel)
    # 'not selected' is either false or unknown; distinguish via the negation.
    false = pm.mask(~sel)
    out = {}
    for i in range(pm.size):
        out[(a_state[i], b_state[i])] = "T" if true[i] else ("F" if false[i] else "U")
    header = "      " + "".join(f"b={k:<4}" for k in "TFU")
    lines = [header]
    for x in "TFU":
        cells = "".join(f"{out[(x, y)]:<6}" for y in "TFU")
        lines.append(f"a={x}   {cells}")
    return "\n".join(lines)


print("AND\n" + table(grid, a["yes"] & b["yes"], a_state, b_state))
print()
print("OR\n" + table(grid, a["yes"] | b["yes"], a_state, b_state))

Read the `AND` table: `T & U` is `U`, not `F` — an unknown cannot be ruled out.
Read the `OR` table: `T | U` is `T` — one true is enough. `F & U` is `F` and
`F | U` is `U`. These are exactly Kleene's strong tables, and they are the
reason ragged maps compose without special cases.

In [ ]:
neg = {}
for i in range(grid.size):
    sel = a["yes"]
    neg[str(a_state[i])] = (
        "T" if grid.mask(~sel)[i] else ("F" if grid.mask(sel)[i] else "U")
    )
assert neg == {"T": "F", "F": "T", "U": "U"}
print("NOT:", neg)

---

Next: {doc}`05-partitions-and-groups` uses these queries to build the index sets
that aggregation and reporting need.